# Домашнее задание по теме "Рекурентные сети 2"

In [1]:
import torch
from torch import nn
import re
import random
import tqdm
import time

## Задание 1

Сгенерировать последовательности, которые бы состояли из цифр (от 0 до 9)
и задавались следующим образом:

x - последовательность цифр

y1 = x1, y(i) = x(i) + x(1). Если y(i) >= 10, то y(i) = y(i) - 10

Задача:
научить модель предсказывать y(i) по x(i)
пробовать RNN, LSTM, GRU

## Генерация последовательности

## Подготовка датасета

## Архитектура нейросети

## Обучение модели

## Задание 2
применить LSTM для решения лекционного практического задания

### Получение и предобработка данных

Первичная предобработка текста

In [2]:
with open('nietzsche.txt', encoding='utf-8') as f:
    text = f.read().lower()
print('length:', len(text))    
text = re.sub('[^a-z]', ' ', text)
text = re.sub('\s+', ' ', text)

length: 600893


Создаем словарь из токенов (символов)

In [3]:
INDEX_TO_CHAR = sorted(list(set(text)))
CHAR_TO_INDEX = {c: i for i, c in enumerate(INDEX_TO_CHAR)}

Генерируем предложения датасета

In [4]:
MAX_LEN = 40
STEP = 3
SENTENCES = []
NEXT_CHARS = []

for i in range(0, len(text) - MAX_LEN, STEP):
    SENTENCES.append(text[i: i + MAX_LEN])
    NEXT_CHARS.append(text[i + MAX_LEN])
print('Num sents:', len(SENTENCES))   

Num sents: 193075


Кодируем токены числами и в pythorch векторы

In [5]:
print('Vectorization...')
X = torch.zeros((len(SENTENCES), MAX_LEN), dtype=int) # размер ваборки * длина предложения
Y = torch.zeros((len(SENTENCES)),dtype=int) # размер выборки

for i, sentence in enumerate(SENTENCES):
    for t, char in enumerate(sentence):
        X[i, t] = CHAR_TO_INDEX[char]
    Y[i] = CHAR_TO_INDEX[NEXT_CHARS[i]] 

Vectorization...


Упаковываем предложения в батчи

In [6]:
BATCH_SIZE = 512
dataset = torch.utils.data.TensorDataset(X, Y)
data = torch.utils.data.DataLoader(
    dataset=dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

### Архитектура нейросери

In [10]:
class NeuralNetwork(nn.Module):
    def __init__(self, rnnClass, dictionary_size, embedding_size, num_hiddens, num_classes):
        super().__init__()

        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(dictionary_size, embedding_size)
        self.hidden = rnnClass(embedding_size, num_hiddens, batch_first=True)
        self.output = nn.Linear(num_hiddens, num_classes)

    def forward(self, X):
        out = self.embedding(X)
        _, state = self.hidden(out)
        prediction = self.output(state[0].squeeze())
        return prediction

Инициализируем модель с LSTM слоем

In [11]:
model = NeuralNetwork(nn.LSTM, len(CHAR_TO_INDEX), 64, 128, len(CHAR_TO_INDEX))

In [12]:
model = model.cuda()

Функция для семплирования токенов

In [13]:
def sample(preds):
    softmaxed = torch.softmax(preds, 0)
    probas = torch.distributions.multinomial.Multinomial(1, softmaxed).sample()
    return probas.argmax()

Функция генерации текста

In [14]:
def generate_text():
    # Берем случайный отрезок текста нужной длины
    start_index = random.randint(0, len(text) - MAX_LEN - 1)

    generated = ''
    sentence = text[start_index: start_index + MAX_LEN]
    generated += sentence

    # Кодируем токены числами
    for i in range(MAX_LEN):
        x_pred = torch.zeros((1, MAX_LEN), dtype=int)
        for t, char in enumerate(generated[-MAX_LEN:]):
            x_pred[0, t] = CHAR_TO_INDEX[char]
        # предсказываем следующий символ
        preds = model(x_pred.cuda()).cpu()
        next_char = INDEX_TO_CHAR[sample(preds)]    
        generated = generated + next_char

    print(generated[:MAX_LEN] + '|' + generated[MAX_LEN:])   

### Обучение модели

In [15]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

In [16]:
for ep in range(30):
    start = time.time()
    train_loss = 0.
    train_passed = 0

    model.train()
    for X_b, y_b in data:
        X_b, y_b = X_b.cuda(), y_b.cuda()
        optimizer.zero_grad()
        answer = model(X_b)
        loss = criterion(answer, y_b)
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        train_passed += 1

    print("Epoch {}. Time: {:.3f}, Train loss: {:.3f}".format(ep, time.time() - start, train_loss / train_passed))    
    model.eval()
    generate_text()

Epoch 0. Time: 2.579, Train loss: 2.166
es of the herding animal things have rea|sto resice all it capatiter istergon tos
Epoch 1. Time: 2.479, Train loss: 1.785
s discerning ones here and there we unde|wrorhers and endurist by prevess and irm
Epoch 2. Time: 2.462, Train loss: 1.651
 light and color into definite figures m|ainst maporhy in everily perhapsed tiswi
Epoch 3. Time: 2.473, Train loss: 1.567
y do so as a form of luxury as an archai|need this sithely hermoss and loveide by
Epoch 4. Time: 2.460, Train loss: 1.508
 of physicians without capacity to attai|nt philosophly sign the been for it uple
Epoch 5. Time: 2.443, Train loss: 1.462
ins may its glance some day overspread l|endom a wish a powiled co speasity of ot
Epoch 6. Time: 2.436, Train loss: 1.426
e anti polish folly the christian romant| more much logs and offerness anadety on
Epoch 7. Time: 2.451, Train loss: 1.396
 very bad taste and we can only evoke in|tellich and we atquent times the molatin
Epoch 8. Time: 2.489, Tr